In [ ]:
import os


project_dir = '/blue/shenhaowang/qingqisong/be-and-active-travel'
os.chdir(project_dir)

In [ ]:
import os
import requests
import math
from PIL import Image
import io
import osmnx as ox
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, Polygon, box
import json
import pickle
import time


class GeoDataCollector:
    def __init__(self, csv_path, distance_miles=0.5, image_resolution=1024,
                 checkpoint_file='point_progress_checkpoint.pkl'):

        self.checkpoint_file = checkpoint_file
        self.metrics_data = {}
        self.processed_ids = set()
        self.load_checkpoint()


        try:
            self.points_df = pd.read_csv(csv_path, sep='\t')
            if len(self.points_df.columns) <= 1:
                self.points_df = pd.read_csv(csv_path, sep=',')
        except Exception:
            self.points_df = pd.read_csv(csv_path)


        self.points_df.columns = self.points_df.columns.str.strip()

        print(f"Loaded {len(self.points_df)} points from {csv_path}")
        print(f"Columns: {list(self.points_df.columns)}")
        print(self.points_df.head())

        # 0.5 mile
        self.distance_m = distance_miles * 1609.34  #(~804.7m)

        self.lat_offset = self.distance_m / 111320.0

        self.image_resolution = image_resolution


        self.directories = {
            'satellite': 'satellite_images_points',
            'osm_maps': 'osm_maps_points',
            'metrics': 'metrics_data_points'
        }
        for directory in self.directories.values():
            os.makedirs(directory, exist_ok=True)

    # ------------------------------------------------------------------ #
    #  Checkpoint
    # ------------------------------------------------------------------ #
    def load_checkpoint(self):

        if os.path.exists(self.checkpoint_file):
            try:
                with open(self.checkpoint_file, 'rb') as f:
                    checkpoint = pickle.load(f)
                    self.processed_ids = checkpoint.get('processed_ids', set())
                    self.metrics_data = checkpoint.get('metrics_data', {})
                    print(f"Restored checkpoint: {len(self.processed_ids)} points already processed")
            except Exception as e:
                print(f"Failed to load checkpoint: {e}")
                self.processed_ids = set()
                self.metrics_data = {}
        else:
            self.processed_ids = set()
            self.metrics_data = {}

    def save_checkpoint(self, point_id):

        try:
            self.processed_ids.add(point_id)
            checkpoint = {
                'processed_ids': self.processed_ids,
                'metrics_data': self.metrics_data
            }
            with open(self.checkpoint_file, 'wb') as f:
                pickle.dump(checkpoint, f)
        except Exception as e:
            print(f"Failed to save checkpoint: {e}")

    # ------------------------------------------------------------------ #
    #  Bounds
    # ------------------------------------------------------------------ #
    def get_tile_bounds(self, lat, lon):
        """

        """
        lat_half = self.lat_offset / 2
        lon_offset = self.lat_offset / math.cos(math.radians(lat))
        lon_half = lon_offset / 2
        return {
            'north': round(lat + lat_half, 6),
            'south': round(lat - lat_half, 6),
            'east': round(lon + lon_half, 6),
            'west': round(lon - lon_half, 6)
        }

    # ------------------------------------------------------------------ #
    #
    # ------------------------------------------------------------------ #
    def download_satellite_image(self, lat, lon, point_id, mapbox_token):

        bounds = self.get_tile_bounds(lat, lon)
        bbox = f"[{bounds['west']},{bounds['south']},{bounds['east']},{bounds['north']}]"

        half_res = self.image_resolution // 2  # 512
        url = (
            f"https://api.mapbox.com/styles/v1/mapbox/satellite-v9/static/"
            f"{bbox}/"
            f"{half_res}x{half_res}"
            f"@2x?access_token={mapbox_token}"
        )

        try:
            response = requests.get(url, timeout=30)
            response.raise_for_status()

            image = Image.open(io.BytesIO(response.content))
            output_path = os.path.join(
                self.directories['satellite'],
                f"satellite_point{point_id}_{round(lat, 6)}_{round(lon, 6)}.png"
            )
            image.save(output_path)
            print(f"  Satellite image saved: {image.size[0]}x{image.size[1]} px")
            return output_path, bounds
        except Exception as e:
            print(f"  Satellite download error (point {point_id}): {e}")
            return None, None

    # ------------------------------------------------------------------ #
    #  OSM
    # ------------------------------------------------------------------ #
    def create_osm_map(self, lat, lon, bounds, point_id):

        try:
            fig, ax = plt.subplots(figsize=(10.24, 10.24))

            landuse_tags = {
                'landuse': [
                    'residential', 'commercial', 'industrial', 'retail',
                    'farmland', 'forest', 'grass', 'meadow', 'orchard',
                    'vineyard', 'cemetery', 'recreation_ground'
                ],
                'leisure': [
                    'park', 'garden', 'golf_course', 'sports_centre',
                    'stadium', 'pitch', 'playground'
                ],
                'natural': [
                    'wood', 'grassland', 'heath', 'scrub', 'wetland',
                    'water', 'beach', 'sand'
                ],
                'amenity': [
                    'school', 'university', 'hospital', 'parking'
                ]
            }

            color_map = {
                'residential': '#FFCA65', 'commercial': '#FF8080',
                'industrial': '#BFBFBF', 'retail': '#FF5555',
                'parking': '#EFEFEF', 'school': '#FFF0AA',
                'university': '#FFF0AA', 'hospital': '#FFE6E6',
                'park': '#B2D8B2', 'garden': '#B2D8B2',
                'recreation_ground': '#B8E6B8', 'playground': '#CCE6CC',
                'sports_centre': '#CCE6CC', 'stadium': '#CCE6CC',
                'pitch': '#CCE6CC', 'golf_course': '#B2D8B2',
                'forest': '#8CBF8C', 'wood': '#8CBF8C',
                'grass': '#CCFFCC', 'grassland': '#CCFFCC',
                'meadow': '#CCFFCC', 'heath': '#CCE6CC',
                'scrub': '#B8E6B8', 'wetland': '#A6E6E6',
                'water': '#B3D9FF', 'beach': '#FFF5CC',
                'sand': '#FFF5CC', 'farmland': '#FFFFCC',
                'orchard': '#E6FFB3', 'vineyard': '#E6FFB3',
                'cemetery': '#D1CFCD'
            }

            for category, values in landuse_tags.items():
                tags = {category: values}
                try:
                    features = ox.geometries_from_bbox(
                        bounds['north'], bounds['south'],
                        bounds['east'], bounds['west'],
                        tags=tags
                    )
                    if not features.empty:
                        for _, row in features.iterrows():
                            feature_type = row[category] if category in row else None
                            if feature_type in color_map:
                                features.loc[[row.name]].plot(
                                    ax=ax,
                                    facecolor=color_map[feature_type],
                                    edgecolor='grey',
                                    alpha=0.7,
                                    linewidth=0.5
                                )
                except Exception:
                    pass

            ax.set_xlim(bounds['west'], bounds['east'])
            ax.set_ylim(bounds['south'], bounds['north'])
            ax.set_axis_off()
            ax.set_facecolor('white')
            fig.patch.set_facecolor('white')

            output_path = os.path.join(
                self.directories['osm_maps'],
                f"osm_map_point{point_id}_{round(lat, 6)}_{round(lon, 6)}.png"
            )
            # dpi=100, figsize=10.24 → 1024 px
            plt.savefig(output_path, dpi=100, bbox_inches='tight', pad_inches=0,
                        facecolor='white')
            plt.close()
            return output_path
        except Exception as e:
            print(f"  OSM map error (point {point_id}): {e}")
            plt.close('all')
            return None

    # ------------------------------------------------------------------ #
    #  metrics
    # ------------------------------------------------------------------ #
    def calculate_metrics(self, lat, lon, bounds):

        try:
            bbox = box(bounds['west'], bounds['south'], bounds['east'], bounds['north'])
            bbox_gdf = gpd.GeoDataFrame(geometry=[bbox], crs='epsg:4326')
            bbox_projected = bbox_gdf.to_crs('epsg:3857')
            area_m2 = bbox_projected.geometry.area.iloc[0]
            area_km2 = area_m2 / 1_000_000


            try:
                buildings = ox.geometries_from_bbox(
                    bounds['north'], bounds['south'],
                    bounds['east'], bounds['west'],
                    tags={'building': True}
                )
            except Exception:
                buildings = gpd.GeoDataFrame()


            roads = None
            try:
                roads = ox.graph_from_bbox(
                    bounds['north'], bounds['south'],
                    bounds['east'], bounds['west'],
                    network_type='all', simplify=True
                )
                if roads is None or len(roads.nodes) == 0 or len(roads.edges) == 0:
                    roads = None
            except Exception:
                roads = None


            land_use_tags = {
                'landuse': ['residential', 'commercial', 'industrial', 'retail',
                            'recreation', 'institutional'],
                'amenity': ['school', 'hospital', 'park', 'university',
                            'library', 'marketplace'],
                'leisure': ['park', 'garden', 'playground']
            }
            try:
                land_uses = ox.geometries_from_bbox(
                    bounds['north'], bounds['south'],
                    bounds['east'], bounds['west'],
                    tags=land_use_tags
                )
            except Exception:
                land_uses = gpd.GeoDataFrame()


            if not buildings.empty:
                buildings_proj = buildings.to_crs('epsg:3857')
                building_area = buildings_proj.geometry.area.sum()
                building_metrics = {
                    'building_density': building_area / area_m2,
                    'building_footprint_ratio': building_area / area_m2,
                    'avg_building_size': buildings_proj.geometry.area.mean()
                }
            else:
                building_metrics = {
                    'building_density': 0,
                    'building_footprint_ratio': 0,
                    'avg_building_size': 0
                }


            road_metrics = {
                'road_density_km': 0,
                'intersection_density': 0,
                'road_network_complexity': 0
            }
            if roads is not None:
                try:
                    edges = ox.graph_to_gdfs(roads, nodes=False)
                    if not edges.empty:
                        edges = edges.to_crs('epsg:3857')
                        total_length = edges.geometry.length.sum()
                        intersection_count = sum(1 for _, deg in roads.degree() if deg > 1)
                        edge_count = len(roads.edges)
                        node_count = len(roads.nodes)
                        road_metrics = {
                            'road_density_km': (total_length / 1000 / area_km2) if area_km2 > 0 else 0,
                            'intersection_density': intersection_count / area_km2 if area_km2 > 0 else 0,
                            'road_network_complexity': edge_count / node_count if node_count > 0 else 0
                        }
                except Exception:
                    pass


            if not land_uses.empty:
                land_uses_proj = land_uses.to_crs('epsg:3857')
                land_use_counts = {}
                land_use_areas = {}
                for category in ['landuse', 'amenity', 'leisure']:
                    if category in land_uses.columns:
                        for land_type, group in land_uses_proj.groupby(land_uses[category]):
                            if land_type and not pd.isna(land_type):
                                key = f"{category}_{land_type}"
                                land_use_counts[key] = len(group)
                                land_use_areas[key] = group.geometry.area.sum()

                avg_land_use_area = (sum(land_use_areas.values()) / len(land_use_areas)
                                     if land_use_areas else 0)

                if land_use_counts:
                    total = sum(land_use_counts.values())
                    proportions = [c / total for c in land_use_counts.values()]
                    shannon_diversity = -sum(p * math.log(p) for p in proportions if p > 0)
                else:
                    shannon_diversity = 0

                land_use_metrics = {
                    'land_use_diversity': shannon_diversity,
                    'unique_land_uses': len(land_use_counts),
                    'avg_land_use_area': avg_land_use_area,
                    'avg_land_use_area_km2': avg_land_use_area / 1_000_000,
                    'amenity_density': (
                        sum(1 for _ in land_uses['amenity'].dropna()) / area_km2
                        if 'amenity' in land_uses.columns else 0
                    )
                }
            else:
                land_use_metrics = {
                    'land_use_diversity': 0,
                    'unique_land_uses': 0,
                    'avg_land_use_area': 0,
                    'avg_land_use_area_km2': 0,
                    'amenity_density': 0
                }


            accessibility_metrics = {
                'avg_distance_to_amenities': 0,
                'nearest_amenity_distance': 0
            }
            if not land_uses.empty and 'amenity' in land_uses.columns:
                amenities = land_uses[land_uses['amenity'].notna()]
                if not amenities.empty:
                    amenities_proj = amenities.to_crs('epsg:3857')
                    center_gdf = gpd.GeoDataFrame(
                        geometry=[Point(lon, lat)], crs='epsg:4326'
                    ).to_crs('epsg:3857')
                    center_point = center_gdf.geometry.iloc[0]
                    distances = [center_point.distance(p) for p in amenities_proj.geometry]
                    if distances:
                        accessibility_metrics = {
                            'avg_distance_to_amenities': sum(distances) / len(distances) / 1000,
                            'nearest_amenity_distance': min(distances) / 1000
                        }

            return {**building_metrics, **road_metrics, **land_use_metrics, **accessibility_metrics}

        except Exception as e:
            print(f"  Metrics calculation error: {e}")
            return {
                'building_density': 0, 'building_footprint_ratio': 0, 'avg_building_size': 0,
                'road_density_km': 0, 'intersection_density': 0, 'road_network_complexity': 0,
                'land_use_diversity': 0, 'unique_land_uses': 0,
                'avg_land_use_area': 0, 'avg_land_use_area_km2': 0, 'amenity_density': 0,
                'avg_distance_to_amenities': 0, 'nearest_amenity_distance': 0
            }

    # ------------------------------------------------------------------ #
    #  point
    # ------------------------------------------------------------------ #
    def process_point(self, lat, lon, point_id, row_data, mapbox_token):

        lat = round(lat, 6)
        lon = round(lon, 6)
        cell_key = f"point_{point_id}"


        if cell_key in self.metrics_data:
            print(f"  Using cached data for point {point_id}")
            cached = self.metrics_data[cell_key]
            return {
                'point_id': point_id,
                'lat': lat, 'lon': lon,
                'n_as_origin': cached.get('n_as_origin', 0),
                'n_as_dest': cached.get('n_as_dest', 0),
                'n_total': cached.get('n_total', 0),
                'satellite_path': os.path.join(
                    self.directories['satellite'],
                    f"satellite_point{point_id}_{lat}_{lon}.png"),
                'osm_path': os.path.join(
                    self.directories['osm_maps'],
                    f"osm_map_point{point_id}_{lat}_{lon}.png"),
                'metrics': cached['metrics']
            }

        print(f"  Processing point {point_id} at ({lat}, {lon})")
        bounds = self.get_tile_bounds(lat, lon)


        satellite_path, _ = self.download_satellite_image(lat, lon, point_id, mapbox_token)

        osm_path = self.create_osm_map(lat, lon, bounds, point_id)

        metrics = self.calculate_metrics(lat, lon, bounds)


        n_as_origin = int(row_data.get('n_as_origin', 0))
        n_as_dest = int(row_data.get('n_as_dest', 0))
        n_total = int(row_data.get('n_total', 0))

        self.metrics_data[cell_key] = {
            'point_id': point_id,
            'lat': lat, 'lon': lon,
            'n_as_origin': n_as_origin,
            'n_as_dest': n_as_dest,
            'n_total': n_total,
            'metrics': metrics
        }


        self.save_checkpoint(point_id)

        return {
            'point_id': point_id,
            'lat': lat, 'lon': lon,
            'n_as_origin': n_as_origin,
            'n_as_dest': n_as_dest,
            'n_total': n_total,
            'satellite_path': satellite_path,
            'osm_path': osm_path,
            'metrics': metrics
        }

    # ------------------------------------------------------------------ #
    #  all
    # ------------------------------------------------------------------ #
    def process_all_points(self, mapbox_token):

        results = []
        total = len(self.points_df)

        for i, (_, row) in enumerate(self.points_df.iterrows()):
            point_id = int(row['point_id'])
            lat = float(row['lat'])
            lon = float(row['lon'])

            print(f"\n=== Point {point_id}  ({i + 1}/{total}) ===")


            if point_id in self.processed_ids:
                cell_key = f"point_{point_id}"
                if cell_key in self.metrics_data:
                    cached = self.metrics_data[cell_key]
                    result = {
                        'point_id': point_id,
                        'lat': cached['lat'], 'lon': cached['lon'],
                        'n_as_origin': cached.get('n_as_origin', 0),
                        'n_as_dest': cached.get('n_as_dest', 0),
                        'n_total': cached.get('n_total', 0),
                        'satellite_path': os.path.join(
                            self.directories['satellite'],
                            f"satellite_point{point_id}_{cached['lat']}_{cached['lon']}.png"),
                        'osm_path': os.path.join(
                            self.directories['osm_maps'],
                            f"osm_map_point{point_id}_{cached['lat']}_{cached['lon']}.png"),
                        'metrics': cached['metrics']
                    }
                    results.append(result)
                    print(f"  Loaded from checkpoint")
                    continue

            result = self.process_point(lat, lon, point_id, row, mapbox_token)
            results.append(result)


            time.sleep(0.5)


        self.save_results(results)
        return results

    # ------------------------------------------------------------------ #
    #  save
    # ------------------------------------------------------------------ #
    def save_results(self, results):

        try:
            metrics_rows = []
            for r in results:
                if r and r.get('metrics'):
                    metrics_rows.append({
                        'point_id': r['point_id'],
                        'lat': r['lat'],
                        'lon': r['lon'],
                        'n_as_origin': r.get('n_as_origin', 0),
                        'n_as_dest': r.get('n_as_dest', 0),
                        'n_total': r.get('n_total', 0),
                        **r['metrics']
                    })

            df = pd.DataFrame(metrics_rows)
            output_csv = os.path.join(self.directories['metrics'], 'metrics_results.csv')
            df.to_csv(output_csv, index=False)
            print(f"\nSaved metrics for {len(metrics_rows)} points → {output_csv}")


            processed_files = {
                'satellite': [r['satellite_path'] for r in results
                              if r and r.get('satellite_path')],
                'osm_maps': [r['osm_path'] for r in results
                             if r and r.get('osm_path')]
            }
            with open('processed_files.json', 'w') as f:
                json.dump(processed_files, f, indent=2)

        except Exception as e:
            print(f"Error saving results: {e}")


# ====================================================================== #
#  mian
# ====================================================================== #
def main():

    csv_path = './od_unique_coordinates_detailed.csv'
    mapbox_token = ''    # Mapbox token


    collector = GeoDataCollector(
        csv_path=csv_path,
        distance_miles=0.5,
        image_resolution=1024,
        checkpoint_file='point_progress_checkpoint.pkl'
    )


    results = collector.process_all_points(mapbox_token)


    print(f"\n{'='*50}")
    print(f"Data collection complete!")
    print(f"Total points processed: {len(results)}")
    print(f"Coverage per point: ~0.5 mile × 0.5 mile "
          f"({collector.distance_m:.0f}m × {collector.distance_m:.0f}m)")
    print(f"Image resolution: {collector.image_resolution}×{collector.image_resolution} px")
    print(f"Output directories:")
    for name, path in collector.directories.items():
        print(f"  {name}: ./{path}/")


if __name__ == "__main__":
    main()

Restored checkpoint: 1148 points already processed
Loaded 1148 points from ./od_unique_coordinates_detailed.csv
Columns: ['lat', 'lon', 'point_id', 'n_as_origin', 'n_as_dest', 'n_total']
         lat        lon  point_id  n_as_origin  n_as_dest  n_total
0  41.891022 -87.612931         1           52         50      102
1  41.928424 -87.684906         2           47         38       85
2  41.895035 -87.619717         3           99        126      225
3  41.776951 -87.623216         4            2          3        5
4  41.865578 -87.630568         5           54         51      105

=== Point 1  (1/1148) ===
  Loaded from checkpoint

=== Point 2  (2/1148) ===
  Loaded from checkpoint

=== Point 3  (3/1148) ===
  Loaded from checkpoint

=== Point 4  (4/1148) ===
  Loaded from checkpoint

=== Point 5  (5/1148) ===
  Loaded from checkpoint

=== Point 6  (6/1148) ===
  Loaded from checkpoint

=== Point 7  (7/1148) ===
  Loaded from checkpoint

=== Point 8  (8/1148) ===
  Loaded from checkp